In [1]:
import numpy as np
import sympy as smp
import matplotlib.pyplot as plt
import pandas as pd

# For interpolation
from scipy.interpolate import RectBivariateSpline, interp2d

import warnings

# Ignore DeprecationWarnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*divmax.*") 

import logging
# Basic registry settings
logging.basicConfig(level=logging.INFO)

In [2]:
import Cosmo_util_data as cu
import Cosmo_integration as ci
import Cosmo_shear as cs

Interpolation of pkz-Fiducial.txt done
Interpolation of pkz-Om_pl_eps_1p3E-2.txt done
Interpolation of pkz-Om_mn_eps_1p3E-2.txt done
Interpolation of pkz-h_pl_eps_1p3E-2.txt done
Interpolation of pkz-h_mn_eps_1p3E-2.txt done
Interpolation of pkz-Ob_pl_eps_1p3E-2.txt done
Interpolation of pkz-Ob_mn_eps_1p3E-2.txt done
Interpolation of pkz-ns_pl_eps_1p3E-2.txt done
Interpolation of pkz-ns_mn_eps_1p3E-2.txt done
Interpolation of pkz-s8_pl_eps_1p3E-2.txt done
Interpolation of pkz-s8_mn_eps_1p3E-2.txt done
Interpolation of luminosity function created.


In [3]:
# Parametros fiduciales

Omega_b0_fid = 0.05
Omega_m0_fid = 0.32
h_fid = 0.67
ns_fid = 0.96
sigma8_fid = 0.816
Omega_DE0_fid = 0.68
w0_fid = -1.0
wa_fid = 0.0
gamma_fid = 0.55

# c = 9.72 * 10 ** (-15) # en Mpc # 300000 en km/s
c = 300000 #en km/s
Aia = 1.72
Cia = 0.0134
nia = -0.41
bia = 2.17


In [4]:
S_Om_m = 0.018
S_h = 0.21
S_Om_b = 0.47
S_ns = 0.035
S_sig = 0.0087

# Esto no sé
S_Aia = 1
S_nia = 1
S_bia = 1

In [5]:
class Fisher:
    '''
    Calculate Fisher Matrix
    '''
    def __init__(self, params):
        self.num = params['num_params']
        self.universes = params['type']
        self.model = params['model']
        self.IA = params['IA']
        self.l_min, self.l_max, self.l_len = params['l']['l_min'], params['l']['l_max'], params['l']['l_len']
        self.z_min, self.z_max, self.z_len = params['zs']['z_min'], params['zs']['z_max'], params['zs']['z_len']

        IA_parametros = ['Aia', 'nia', 'bia']
        IA_parametros_values = [Aia, nia, bia]
        IA_results_euclid = [S_Aia, S_nia, S_bia]
        if self.model == 'ACDM_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8"]
            parametros_values = [Omega_m0_fid, h_fid, Omega_b0_fid, ns_fid, sigma8_fid]
            results_euclid = [S_Om_m, S_h, S_Om_b, S_ns, S_sig]
        elif self.model == 'ACDM_non_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'Omega_DE0']
        elif self.model == 'non_ACDM_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'w0', 'wa']
        elif self.model == 'non_ACDM_non_flat':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'Omega_DE0', 'w0', 'wa']
        elif self.model == 'non_ACDM_flat_gamma':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'w0', 'wa', 'gamma']       
        elif self.model == 'non_ACDM_non_flat_gamma':
            parametros = ["Omega_m0", "h", "Omega_b0", "ns", "sigma8", 'Omega_DE0', 'w0', 'wa', 'gamma']       

        if self.IA == True:
            parametros = parametros + IA_parametros
            parametros_values = parametros_values + IA_parametros_values
            results_euclid = results_euclid + IA_results_euclid
        
        self.parametros = parametros
        self.parametros_values = parametros_values
        self.results_euclid = results_euclid

    def trace(self, Omega_m0, h, Omega_b0, Omega_DE0, w0, wa, ns, sigma8, gamma, Aia, nia, bia):
        if self.IA == True:
            F = np.zeros((self.num + 3, self.num + 3))
        else:
            F = np.zeros((self.num, self.num))

        L_array = np.log10(np.logspace(np.log10(self.l_min), np.log10(self.l_max), self.l_len))
        zs = np.linspace(self.z_min, self.z_max, self.z_len) #np.logspace(np.log10(self.z_min), np.log10(self.z_max), self.z_len)
        cosmic_parametros = {'l': L_array , 'z': zs, 'type': self.universes, 'model': self.model, 'IA': self.IA}

        A = cs.CosmicShear(cosmic_parametros)  
        f_sky = 0.3636
        epsilon = 0.013

        size = 10  

        def derivative(i, j, param):
            deriv = A.Der_C_parametro(i ,j, epsilon, Omega_m0, h, Omega_b0, Omega_DE0, w0, wa, ns, sigma8, gamma, Aia, nia, bia, param)
            return deriv
        
        def Cosmic_Shear(i, j):
            CS = A.Cosmic_Shear(i, j, Omega_m0, h, Omega_b0, Omega_DE0, w0, wa, ns, sigma8, gamma, Aia, nia, bia)
            return CS

        # Crear un diccionario con todas las matrices
        C = np.zeros((size, size), dtype=object)
        dC_dq_matrices = {p: np.zeros((size, size), dtype=object) for p in self.parametros}
        
        for i in range(size):
            for j in range(i, size):  # solo parte triangular superior
                logging.info(f"Calculated pair: {(i, j)}")
                C[i, j] = C[j, i] = Cosmic_Shear(i, j)
                for p in self.parametros:
                    val = derivative(i, j, p)
                    dC_dq_matrices[p][i, j] = val
                    dC_dq_matrices[p][j, i] = val
        
        C_matrix = np.array(C.tolist(), dtype=float)
        C_matrix_2d = np.squeeze(C_matrix)

        dC_dq_matrices_2d = {}
        for p in self.parametros:
            mat = np.array(dC_dq_matrices[p].tolist(), dtype=float)
            dC_dq_matrices_2d[p] = np.squeeze(mat)

        for i, l in enumerate(L_array):

            # definiciones de lambda_k y delta_l como ya las tienes
            def lambda_k(i): 
                lambda_min = np.log10(10**L_array[0])
                lambda_max = np.log10(10**L_array[-1])
                delta_lambda = (lambda_max - lambda_min) / len(L_array)
                return lambda_min + (i - 1)*delta_lambda
            
            delta_l = 10**lambda_k(i + 1) - 10**lambda_k(i)

            L_parameter_new = ((2*(10 ** l) + 1) * delta_l * f_sky) / 2 
            
            Ci = C_matrix_2d[:, :, i]
            C_inv  = np.linalg.inv(Ci)

            coeficientes = {}
            # Loop automático sobre pares de parámetros
            for a, p in enumerate(self.parametros):
                for b, q in enumerate(self.parametros[a:], start=a):
                    mat_p = dC_dq_matrices_2d[p][:, :, i]
                    mat_q = dC_dq_matrices_2d[q][:, :, i]
                    val   = np.trace(C_inv @ mat_p @ C_inv @ mat_q)
                    coeficientes[(a, b)] = val  

            # Rellena la Fisher matrix
            for (i, j), coef in coeficientes.items():
                F[i, j] += (L_parameter_new * coef)

        # Simetriza
        Fisher = F + F.T - np.diag(F.diagonal())

        logging.info(f"Fisher Matrix: {Fisher}")
        return Fisher
    
    def Covarianzas(self, F):
        Cov = np.linalg.inv(F)
        return Cov
    
    def results(self, Cov):
        def comparison(created, expected):
            return 100*np.abs(1 - (created/expected))
        errors = {}
        errors_relative = {}
        per = {}
        for i, p in enumerate(self.parametros):
            errors[p] = str(np.sqrt(Cov[i, i]))
            errors_relative[p] = (np.sqrt(Cov[i, i]) / self.parametros_values[i])
            per[p] = comparison(errors_relative[p], self.results_euclid[i])
        return errors_relative, per

In [6]:
params = {'num_params': 5, 'type': 'standard', 'model' : 'ACDM_flat', 'IA': True, 'l': {'l_min': 10, 'l_max': 1500, 'l_len': 100}, 
          'zs': {'z_min': 0.001, 'z_max': 2.5, 'z_len': 20}}

A = Fisher(params)

In [7]:
F = A.trace(Omega_m0_fid, h_fid, Omega_b0_fid, Omega_DE0_fid, w0_fid, wa_fid, ns_fid, sigma8_fid, gamma_fid, Aia, nia, bia)

INFO:root:Calculated pair: (0, 0)
INFO:root:Calculated pair: (0, 1)
INFO:root:Calculated pair: (0, 2)
INFO:root:Calculated pair: (0, 3)
INFO:root:Calculated pair: (0, 4)
INFO:root:Calculated pair: (0, 5)
INFO:root:Calculated pair: (0, 6)
INFO:root:Calculated pair: (0, 7)
INFO:root:Calculated pair: (0, 8)
INFO:root:Calculated pair: (0, 9)
INFO:root:Calculated pair: (1, 1)
INFO:root:Calculated pair: (1, 2)
INFO:root:Calculated pair: (1, 3)
INFO:root:Calculated pair: (1, 4)
INFO:root:Calculated pair: (1, 5)
INFO:root:Calculated pair: (1, 6)
INFO:root:Calculated pair: (1, 7)
INFO:root:Calculated pair: (1, 8)
INFO:root:Calculated pair: (1, 9)
INFO:root:Calculated pair: (2, 2)
INFO:root:Calculated pair: (2, 3)
INFO:root:Calculated pair: (2, 4)
INFO:root:Calculated pair: (2, 5)
INFO:root:Calculated pair: (2, 6)
INFO:root:Calculated pair: (2, 7)
INFO:root:Calculated pair: (2, 8)
INFO:root:Calculated pair: (2, 9)
INFO:root:Calculated pair: (3, 3)
INFO:root:Calculated pair: (3, 4)
INFO:root:Calc

In [8]:
Cov = A.Covarianzas(F)

In [9]:
Cov

array([[ 1.19390651e-06,  1.18135722e-07,  4.45504647e-07,
         1.76700078e-07, -2.07448292e-06, -1.61678420e-02,
         6.57493538e-03, -6.31125412e-03],
       [ 1.18135722e-07,  4.23607513e-07,  2.42269398e-07,
         1.04908235e-06, -6.43994592e-07, -2.43451156e-03,
        -4.46582635e-04, -1.98628100e-03],
       [ 4.45504647e-07,  2.42269398e-07,  3.61728636e-07,
         1.33490640e-06, -1.44415505e-06, -6.91156927e-03,
         1.67175730e-03, -3.21550349e-03],
       [ 1.76700078e-07,  1.04908235e-06,  1.33490640e-06,
         1.03879492e-05, -7.52315064e-06, -1.79649308e-02,
         8.43172618e-04, -7.73750397e-03],
       [-2.07448292e-06, -6.43994592e-07, -1.44415505e-06,
        -7.52315064e-06,  9.94009669e-06,  3.72577568e-02,
        -1.05203167e-02,  1.39054475e-02],
       [-1.61678420e-02, -2.43451156e-03, -6.91156928e-03,
        -1.79649309e-02,  3.72577568e-02,  3.46307046e+03,
        -2.17545826e+03,  8.91666674e+02],
       [ 6.57493539e-03, -4.465826

In [10]:
rel, perct = A.results(Cov)

In [11]:
rel

{'Omega_m0': np.float64(0.003414563405972829),
 'h': np.float64(0.0009714199634409227),
 'Omega_b0': np.float64(0.012028776090370893),
 'ns': np.float64(0.0033573272550551226),
 'sigma8': np.float64(0.0038637155501086177),
 'Aia': np.float64(34.21387128930216),
 'nia': np.float64(-90.92048835679128),
 'bia': np.float64(7.101744640450817)}

In [12]:
perct

{'Omega_m0': np.float64(81.03020330015094),
 'h': np.float64(99.53741906502813),
 'Omega_b0': np.float64(97.44068593821896),
 'ns': np.float64(90.40763641412822),
 'sigma8': np.float64(55.58947643553312),
 'Aia': np.float64(3321.387128930216),
 'nia': np.float64(9192.048835679128),
 'bia': np.float64(610.1744640450817)}

In [13]:
#Con normalizacion

{'Omega_m0': np.float64(0.005005899963608804),
 'h': np.float64(0.001248654205106214),
 'Omega_b0': np.float64(0.018459575106129247),
 'ns': np.float64(0.0035969432211091076),
 'sigma8': np.float64(0.008443202128281964),
 'Aia': np.float64(42.5444761031612),
 'nia': np.float64(-112.7001449718042),
 'bia': np.float64(8.722090020454557)}

# Sin normalizacion

{'Omega_m0': np.float64(0.0006862816600103079),
 'h': np.float64(0.0003382409342869184),
 'Omega_b0': np.float64(0.0024279972635347655),
 'ns': np.float64(0.0004041128604146491),
 'sigma8': np.float64(7.056934487180305e-05),
 'Aia': np.float64(0.07558649968950613),
 'nia': np.float64(-0.20664905506696646),
 'bia': np.float64(0.014502386264304867)}

{'Omega_m0': np.float64(0.0006862816600103079),
 'h': np.float64(0.0003382409342869184),
 'Omega_b0': np.float64(0.0024279972635347655),
 'ns': np.float64(0.0004041128604146491),
 'sigma8': np.float64(7.056934487180305e-05),
 'Aia': np.float64(0.07558649968950613),
 'nia': np.float64(-0.20664905506696646),
 'bia': np.float64(0.014502386264304867)}